# Part B — A long wave over variable bathymetry

**Working time:** one 90-minute block
**Work in groups of 2–4.** This notebook is guided and is not submitted.

Part B investigates a tsunami-like long disturbance. It is not a hazard
or inundation model. Every grid cell remains wet, including the final
coastal cell.

## Learning goals

- Relate local long-wave speed to local water depth.
- Use virtual gauges to compare arrival time and surface elevation.
- Conduct one controlled bathymetry experiment.
- Observe wind setup and the free response after wind shut-off.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, animate_eta, backend_info, compute_dt_cfl, depth_on_u,
    make_grid, run_model, shelf_bathymetry, uniform_wind_forcing,
    zero_forcing,
)

print(backend_info())


def find_course_root():
    candidates = (
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "MT1562_python_lab_waves",
    )
    return next(path for path in candidates if (path / "notebooks").exists())


COURSE_ROOT = find_course_root()
ANIMATION_DIR = COURSE_ROOT / "animations"
ANIMATION_DIR.mkdir(exist_ok=True)


In [ ]:
def animate_and_save(out, grid, filename, *, title, max_frames=42):
    # Display a compact eta animation and save the same frames as a GIF.
    frame_count = len(out["time"])
    frames = np.unique(
        np.linspace(0, frame_count - 1, min(max_frames, frame_count), dtype=int)
    )
    animation = animate_eta(
        out, grid, frames=frames, interval=100, repeat=True,
        title=title, contours=False, remove_mean=False,
        figsize=(8.5, 3.6),
    )
    path = ANIMATION_DIR / filename
    animation.save(str(path), fps=10, dpi=85)
    print("Saved animation:", path.resolve())
    return animation


## 1. Build a shelf and make a prediction

The basin is deep in the west and shallow near the eastern wall. The
disturbance is broad and nearly one-dimensional, which reduces geometric
spreading and makes the depth effect easier to isolate.


In [ ]:
Nx, Ny = 160, 24
Lx, Ly = 2.4e6, 360e3
grid = make_grid(Nx, Ny, Lx, Ly)
H = shelf_bathymetry(
    grid, H_deep=3000.0, H_coast=120.0,
    shelf_width=800e3, coast="east", power=1.5,
)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(grid.x_c / 1e3, H[Ny // 2], linewidth=2)
ax.invert_yaxis()
ax.set(xlabel="x [km]", ylabel="water depth [m]", title="Model bathymetry")
ax.grid(alpha=0.25)
plt.show()

c_deep = np.sqrt(9.81 * 3000.0)
c_coast = np.sqrt(9.81 * 120.0)
print(f"deep-water long-wave speed: {c_deep:.1f} m/s")
print(f"coastal long-wave speed:    {c_coast:.1f} m/s")


**Prediction 1.** Where will the wave travel fastest? What changes do you expect as it crosses the shelf?

**Your response:**


In [ ]:
def cross_basin_pulse(grid, params, *, amplitude=0.12, radius=80e3, x0=350e3):
    eta_line = amplitude * np.exp(-((grid.x_c - x0) / radius) ** 2)
    eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)
    H_u = depth_on_u(grid, params.H)
    eta_u = amplitude * np.exp(-((grid.x_u - x0) / radius) ** 2)
    u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g / H_u)
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v


def run_shelf_case(H_coast, *, tmax_hours=7.0):
    depth = shelf_bathymetry(
        grid, H_deep=3000.0, H_coast=H_coast,
        shelf_width=800e3, coast="east", power=1.5,
    )
    params = ModelParams(H=depth, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True)
    dt = compute_dt_cfl(grid, params, cfl=0.42)
    out = run_model(
        tmax=tmax_hours * 3600,
        dt=dt,
        grid=grid,
        params=params,
        forcing_fn=zero_forcing,
        ic_fn=lambda g, p: cross_basin_pulse(g, p),
        save_every=5,
        out_vars=("eta",),
    )
    return params, out


params_120, out_120 = run_shelf_case(120.0)
eta_120 = np.asarray(out_120["eta"])
times_120 = np.asarray(out_120["time"])
eta_line_120 = eta_120.mean(axis=1)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(
    grid.x_c / 1e3, times_120 / 3600, eta_line_120,
    shading="auto", cmap="RdBu_r"
)
ax.set(xlabel="x [km]", ylabel="time [hours]", title="Wave crossing the shelf")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


In [ ]:
shelf_animation = animate_and_save(
    out_120, grid, "part_b_shelf_120m.gif",
    title="Long wave crossing a shelf: coastal depth = 120 m",
)
shelf_animation


### A dispersive-looking wake — what is it?

The leading pulse develops a wavetrain as it crosses the variable bottom.
That is worth noticing: uniform-depth *linear shallow-water theory* is
non-dispersive because all long wavelengths have speed $c=\sqrt{gH}$.
Here the changing bathymetry scatters and partially reflects the wave, so
different spatial components interfere and the signal spreads. The
finite-difference grid can also add **numerical dispersion**, especially
for features represented by only a few cells. This model does not contain
the full finite-depth surface-wave dispersion relation from the lecture.

A useful diagnostic is to repeat the case with a broader initial pulse or
a finer grid. A wake that changes strongly with resolution is numerical;
a robust wake tied to the shelf is evidence of topographic scattering.


**Observation 1.** In the animation, where does the pulse first develop a visible wake, and which two mechanisms could contribute?

**Your response:**


## 2. Virtual gauges

Four gauges sample the deep basin, the shelf break, the slope, and the
coastal region. The largest peak is not always the first arrival, so use
both the Hovmöller diagram and the time series.


In [ ]:
gauge_x_km = [700, 1400, 1900, 2250]
gauge_indices = [int(np.argmin(abs(grid.x_c / 1e3 - x))) for x in gauge_x_km]

fig, ax = plt.subplots(figsize=(9, 4))
for x_km, index in zip(gauge_x_km, gauge_indices):
    ax.plot(times_120 / 3600, eta_line_120[:, index], label=f"{x_km} km")
ax.set(xlabel="time [hours]", ylabel="surface displacement [m]", title="Virtual gauges")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.show()

peak_times = []
for x_km, index in zip(gauge_x_km, gauge_indices):
    signal = eta_line_120[:, index]
    peak_index = int(np.argmax(signal))
    peak_times.append(times_120[peak_index] / 3600)
    print(f"gauge {x_km:4.0f} km: largest positive peak at {peak_times[-1]:.2f} h, "
          f"eta={signal[peak_index]:.3f} m")


**Analysis 1.** Use the plots and gauge records to explain how depth affected propagation. Include at least one number.

**Your response:**


## 3. Controlled coastal-depth experiment

Repeat the case with a 300 m coastal depth. Everything else remains the
same. Compare arrival time and the modeled elevation at the final gauge.


**Prediction 2.** Will the 300 m coastal case arrive earlier or later than the 120 m case? What do you expect for elevation?

**Your response:**


In [ ]:
params_300, out_300 = run_shelf_case(300.0)
eta_line_300 = np.asarray(out_300["eta"]).mean(axis=1)
times_300 = np.asarray(out_300["time"])
coastal_index = gauge_indices[-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_120 / 3600, eta_line_120[:, coastal_index], label="coastal depth 120 m")
ax.plot(times_300 / 3600, eta_line_300[:, coastal_index], label="coastal depth 300 m")
ax.set(xlabel="time [hours]", ylabel="surface displacement [m]",
       title="Near-coast model cell: controlled comparison")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

for label, times, signal in (
    ("120 m", times_120, eta_line_120[:, coastal_index]),
    ("300 m", times_300, eta_line_300[:, coastal_index]),
):
    index = int(np.argmax(signal))
    print(f"{label}: peak time={times[index]/3600:.2f} h, peak eta={signal[index]:.3f} m")


In [ ]:
shelf_300_animation = animate_and_save(
    out_300, grid, "part_b_shelf_300m.gif",
    title="Controlled comparison: coastal depth = 300 m",
)
shelf_300_animation


**Analysis 2.** Summarize the controlled comparison. Why must it not be interpreted as a prediction of coastal danger?

**Your response:**


## 4. Short wind-forced demonstration

Here the model starts from rest. An eastward wind ramps up, is switched
off after six hours, and pushes water toward the eastern wall.


In [ ]:
wind_params = ModelParams(
    H=H, g=9.81, f0=0.0, beta=0.0,
    r=1/(2*86400), linear=True,
)
wind_dt = compute_dt_cfl(grid, wind_params, cfl=0.42)
wind_forcing = lambda t, g, p: uniform_wind_forcing(
    t, g, p, tau_x=0.08, tau_y=0.0,
    t_ramp=2*3600, t_off=6*3600,
)
wind_out = run_model(
    tmax=8*3600, dt=wind_dt, grid=grid, params=wind_params,
    forcing_fn=wind_forcing,
    ic_fn=lambda g, p: (
        np.zeros((g.Ny, g.Nx)),
        np.zeros((g.Ny, g.Nx+1)),
        np.zeros((g.Ny+1, g.Nx)),
    ),
    save_every=8, out_vars=("eta",),
)
wind_eta = np.asarray(wind_out["eta"])
wind_times = np.asarray(wind_out["time"])
shutoff_index = int(np.argmin(abs(wind_times - 6*3600)))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(
    grid.x_c/1e3, wind_eta[shutoff_index].mean(axis=0),
    label="near wind shut-off (setup)",
)
ax.plot(
    grid.x_c/1e3, wind_eta[-1].mean(axis=0),
    label="two hours later (free response)",
)
ax.axhline(0, color="0.4", linewidth=0.8)
ax.set(xlabel="x [km]", ylabel="surface displacement [m]",
       title="Wind setup and release")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


In [ ]:
wind_animation = animate_and_save(
    wind_out, grid, "part_b_wind_setup_release.gif",
    title="Wind setup followed by a free basin response",
)
wind_animation


**Observation 2.** Which coast gains water under eastward wind? What happens after the wind stops?

**Your response:**
